# Procurement transaction KPIs with Databricks SQL

## Summary
The original saved queries report **5,804,212 transactions** and **$653.865 billion in summed federal action obligations** for the supplied snapshot. These values describe this extraction, not an independently reconciled government-wide total.

## Context and methods
Julio Hernandez defined the KPIs and wrote the original SQL. This portfolio edition retains six transaction-level queries. Their calculations and saved outputs are unchanged. Historical aliases such as `number_of_contracts` remain in code for traceability; the narrative correctly identifies them as transaction counts. Four exploratory queries with unresolved interpretation issues are excluded and documented in [limitations](../docs/limitations.md).

### Assumptions and setup
Run `01_data_preparation.ipynb` first in Databricks. Replace `workspace.usaspending` with your project namespace. `%sql` requires Databricks; these cells do not execute in an ordinary Python kernel. Saved outputs came from the original notebooks and have not been rerun for this portfolio edition.

## Results

### Total transaction obligations

Sum `federal_action_obligation` over transaction records and express in USD billions. Negative adjustments remain included. Obligations are commitments, not cash outlays.

In [ ]:
%sql
SELECT 
  SUM(federal_action_obligation) / '1e9' AS total_federal_action_obligation
FROM workspace.usaspending.fy2025_all_contracts_bronze

total_federal_action_obligation
653.8648378448692


### Procurement transaction volume

Count rows, expressed in millions. The historical SQL alias says contracts; it is a transaction count, not a distinct-award count.

In [ ]:
%sql
SELECT 
  COUNT(*) / 1000000 AS number_of_contracts
FROM workspace.usaspending.fy2025_all_contracts_bronze

number_of_contracts
5.804212


### Top agencies by transaction obligations

Rank the top 10 agencies by summed transaction obligations, in USD. These rows do not constitute the full agency universe.

In [ ]:
%sql
SELECT 
  awarding_agency_name, 
  SUM(federal_action_obligation) AS total_obligation
FROM workspace.usaspending.fy2025_all_contracts_bronze
GROUP BY awarding_agency_name
ORDER BY total_obligation DESC
LIMIT 10

awarding_agency_name,total_obligation
Department of Defense,3.609220375087587E11
Department of Veterans Affairs,7.060907216044002E10
Department of Energy,5.026198135261997E10
Department of Homeland Security,2.833970658616996E10
Department of Health and Human Services,2.676471360212992E10
General Services Administration,2.469205032021989E10
National Aeronautics and Space Administration,2.120567413860002E10
Department of Transportation,9.87587888324001E9
Department of Agriculture,9.594317272570005E9
Department of State,9.480012331380024E9


### Full and open competition category share

Percentage of all transaction rows with the exact recorded category `FULL AND OPEN COMPETITION`. Other competition categories and missing values remain in the denominator. This is not a comprehensive competed-award rate.

In [ ]:
%sql
SELECT 
  ROUND(
    100.0 * SUM(CASE WHEN extent_competed = 'FULL AND OPEN COMPETITION' THEN 1 ELSE 0 END) / COUNT(*), 
    2
  ) AS percent_competed
FROM workspace.usaspending.fy2025_all_contracts_bronze

percent_competed
59.38


### Transaction distribution by award type

Count transactions by the recorded award-type field, retaining missing categories. Award type is not the same as pricing type. The historical output alias `contract_count` means transaction rows.

In [ ]:
%sql
SELECT 
  award_type, 
  COUNT(*) AS contract_count
FROM workspace.usaspending.fy2025_all_contracts_bronze
GROUP BY award_type
ORDER BY contract_count DESC

award_type,contract_count
DELIVERY ORDER,3804384
BPA CALL,807131
PURCHASE ORDER,749635
null,297943
DEFINITIVE CONTRACT,145073
AWARD,28
IDV,9
GA-12,3
GA-02,3
B,2


### Top industries by transaction obligations

Group transaction counts and obligations by NAICS description, then return the 10 highest-obligation groups. Historical `contract_count` aliases mean transactions.

In [ ]:
%sql
SELECT 
  naics_description, 
  COUNT(*) AS contract_count, 
  SUM(federal_action_obligation) AS total_obligation
FROM workspace.usaspending.fy2025_all_contracts_bronze
GROUP BY naics_description
ORDER BY total_obligation DESC
LIMIT 10

naics_description,contract_count,total_obligation
ENGINEERING SERVICES,78994,4.336857177765995E10
"RESEARCH AND DEVELOPMENT IN THE PHYSICAL, ENGINEERING, AND LIFE SCIENCES (EXCEPT NANOTECHNOLOGY AND BIOTECHNOLOGY)",46152,4.0629534443430046E10
AIRCRAFT MANUFACTURING,27263,3.971061163717996E10
DIRECT HEALTH AND MEDICAL INSURANCE CARRIERS,2056,3.889222069186001E10
FACILITIES SUPPORT SERVICES,32877,3.649743068853001E10
SHIP BUILDING AND REPAIRING,22707,3.383516075605002E10
COMPUTER SYSTEMS DESIGN SERVICES,44100,2.8798982952530014E10
COMMERCIAL AND INSTITUTIONAL BUILDING CONSTRUCTION,40100,2.332879952498999E10
OTHER COMPUTER RELATED SERVICES,77327,2.2388172002520058E10
GUIDED MISSILE AND SPACE VEHICLE MANUFACTURING,2652,2.208673950014E10


## Takeaways
The SQL demonstrates aggregation, conditional calculations, category segmentation, ranking, and clear metric definitions. Spending and activity are different measures: a high-volume category need not have the highest obligations. The source snapshot and transaction grain constrain interpretation.

Open [the executable results showcase](03_results_showcase.ipynb) for charts built from these saved aggregates. That notebook can run locally without access to the original Databricks workspace.